# QuantJourney SDK - Brinson Benchmark-Relative Attribution

This notebook demonstrates a QuantJourney SDK workflow that calculates allocation, selection and interaction effects using holdings, benchmark weights and sector return buckets.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
sector_feed = qj.yf.get_sp500_sectors()

def normalize_sector_map(payload: Any) -> dict[str, str]:
    rows = as_rows(payload)
    sector_map = {}
    for item in rows:
        symbol = item.get('symbol') or item.get('ticker') or item.get('Symbol')
        sector = item.get('sector') or item.get('Sector') or item.get('gics_sector')
        if symbol and sector:
            sector_map[str(symbol).upper()] = str(sector)
    return sector_map
sectors = normalize_sector_map(sector_feed)
portfolio_w = pd.Series({'AAPL': 0.18, 'MSFT': 0.18, 'NVDA': 0.16, 'GOOGL': 0.12, 'AMZN': 0.12, 'JPM': 0.1, 'XOM': 0.06, 'LLY': 0.08})
benchmark_w = pd.Series({'AAPL': 0.12, 'MSFT': 0.12, 'NVDA': 0.1, 'GOOGL': 0.09, 'AMZN': 0.09, 'JPM': 0.08, 'XOM': 0.1, 'LLY': 0.08})
benchmark_w = benchmark_w / benchmark_w.sum()
prices, volumes = price_panel(list(portfolio_w.index))
period_return = prices.iloc[-1] / prices.iloc[0] - 1


In [ ]:
df = pd.DataFrame({'sector': pd.Series(sectors), 'portfolio_w': portfolio_w, 'benchmark_w': benchmark_w, 'return': period_return})
df['sector'] = df['sector'].fillna('Unknown')
sector = df.groupby('sector').apply(lambda x: pd.Series({'portfolio_w': x['portfolio_w'].sum(), 'benchmark_w': x['benchmark_w'].sum(), 'portfolio_return': np.average(x['return'], weights=x['portfolio_w'] / x['portfolio_w'].sum()), 'benchmark_return': np.average(x['return'], weights=x['benchmark_w'] / x['benchmark_w'].sum())}))
benchmark_total = float((df['benchmark_w'] * df['return']).sum())
sector['allocation'] = (sector['portfolio_w'] - sector['benchmark_w']) * (sector['benchmark_return'] - benchmark_total)
sector['selection'] = sector['benchmark_w'] * (sector['portfolio_return'] - sector['benchmark_return'])
sector['interaction'] = (sector['portfolio_w'] - sector['benchmark_w']) * (sector['portfolio_return'] - sector['benchmark_return'])
display(sector)
sector[['allocation', 'selection', 'interaction']].plot(kind='bar', stacked=True, title='Brinson attribution')
plt.ylabel('Contribution')
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.